# Original lifetime Oλ estimates and particles
Fit the original log-excess trajectory model to the observed lifetime statistic. Use the fitted posterior scale for each component, with zero additive calibration floor. Save 16 equal-weight asymptote particles per patient for survival analysis.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import tempfile

import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
DATA = Path('/content/drive/MyDrive/Hartert/CAR-T-spectral-organization/DATA')
SOURCE = DATA / 'olambda_asymptotes_min1000'
TRAJECTORIES = SOURCE / 'combined_olambda_trajectory_points.csv'
COHORT = SOURCE / 'patient_olambda_asymptotes.csv'
OUTPUT = DATA / 'olambda_asymptotes_min1000_no_floor_v1'
for path in (TRAJECTORIES, COHORT):
    if not path.is_file():
        raise FileNotFoundError(path)

In [ ]:
# Use the validated revision in an isolated checkout, including on a reused runtime.
repo = Path(tempfile.mkdtemp(prefix='direct-olambda-')) / 'repo'
revision = '42949bfb686819b08054293d42bea6e671452a29'
subprocess.run(['git', 'clone', '--quiet', '--single-branch', '--branch', 'olambda-lifetime-no-floor',
                'https://github.com/adamnik/CAR-T-spectral-organization.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--quiet', '--detach', revision], check=True)
if shutil.which('Rscript') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', 'r-base', 'r-cran-nlme'], check=True)
subprocess.run(['Rscript', '-e', 'if (!requireNamespace("nlme", quietly=TRUE)) install.packages("nlme", repos="https://cloud.r-project.org"); stopifnot(requireNamespace("nlme", quietly=TRUE))'], check=True)
print('Estimator revision:', revision)

In [ ]:
keys = ['Dataset', 'sample_id']
cohort = pd.read_csv(COHORT, usecols=keys)
assert len(cohort) == 163 and cohort.Dataset.nunique() == 8
assert not cohort.duplicated(keys).any() and cohort.notna().all().all()
points = pd.read_csv(TRAJECTORIES, usecols=keys + ['n_cells_available', 'target_depth', 'O_lambda'])
points = points.merge(cohort, on=keys, validate='many_to_one')
eligible = points.loc[(points.n_cells_available >= 1000) & np.isfinite(points.target_depth)
                      & np.isfinite(points.O_lambda) & (points.O_lambda > 1 + 1e-5)]
assert len(eligible) == 20992 and len(eligible[keys].drop_duplicates()) == len(cohort)
display(eligible.groupby('Dataset').agg(patients=('sample_id', 'nunique'), points=('target_depth', 'size')))

In [ ]:
import hashlib
import json

est_path = OUTPUT / 'patient_olambda_asymptotes.csv'
particle_path = OUTPUT / 'patient_olambda_particles.csv'
with tempfile.TemporaryDirectory() as local:
    subset = Path(local) / 'trajectory.csv'
    eligible.to_csv(subset, index=False, float_format='%.17g')
    config = dict(revision=revision, trajectory_sha256=hashlib.sha256(subset.read_bytes()).hexdigest(),
                  minimum_cells=1000, particles=16, seed=0)
    if OUTPUT.exists():
        if json.loads((OUTPUT / 'notebook_config.json').read_text()) != config:
            raise ValueError(f'Existing output has different inputs or settings: {OUTPUT}')
    else:
        stage = Path(tempfile.mkdtemp(prefix='lifetime_fit_', dir=OUTPUT.parent))
        subprocess.run(['Rscript', repo / 'spectral_organization/estimation/fit_olambda_asymptotes.R',
                        subset, stage, '--minimum-cells', '1000', '--n-particles', '16',
                        '--random-state', '0'], check=True)
        summary = pd.read_csv(stage / 'population_model_summary.csv').iloc[0]
        assert summary.calibrated_log_asymptote_floor == 0 and summary.trajectory_points == len(eligible)
        for file in (est_path.name, particle_path.name, 'population_model.rds'):
            assert (stage / file).stat().st_size > 0
        (stage / 'notebook_config.json').write_text(json.dumps(config, indent=2))
        stage.rename(OUTPUT)


In [ ]:
est = pd.read_csv(est_path).set_index(keys, verify_integrity=True).sort_index()
pt = pd.read_csv(particle_path).set_index(keys, verify_integrity=True).sort_index()
expected = cohort.set_index(keys).sort_index().index
assert est.index.equals(expected) and pt.index.equals(expected)
assert est.model.eq('latent_timescale_student_t3_scale_0.35_no_asymptote_floor').all()
assert list(pt.columns) == [f'particle_{k:02d}' for k in range(1, 17)]
a = pt.to_numpy()
assert ((a > 1) & np.isfinite(a)).all() and (np.diff(a, axis=1) >= 0).all()
assert (est.O_lambda_inf_q025.gt(1) & est.O_lambda_inf_q975.gt(est.O_lambda_inf_q025)).all()
assert est.n_trajectory_points_available.sum() == len(eligible)
print(f'{len(est)} patients | {est.index.get_level_values("Dataset").nunique()} datasets | {pt.shape[1]} particles')
print('Asymptotes:', est_path)
print('Survival particles:', particle_path)
display(est[['n_cells_available', 'O_lambda_inf_central', 'O_lambda_inf_q025', 'O_lambda_inf_q975']].head(10))